In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def beam_splitter(r):
    """
    Returns the beam splitter matrix.

    Args:
        - r (float): The reflection coefficient of the beam splitter.
    Returns:
        - (np.array): 2 x 2 matrix that represents the beam
        splitter matrix.    
    """
    t = np.sqrt(1 - r**2)
    return np.array([[r, t], [t, -r]])


dev = qml.device('default.qubit', wires=2, shots=None) # Use 2 wires for the simulation, shots=None for exact probabilities

@qml.qnode(dev)
def mz_interferometer(r):
    """
    This QNode returns the probability that either A or C
    detect a photon, and the probability that D detects a photon.
    
    Args:
        - r (float): The reflection coefficient of the beam splitters.
    Returns: 
        - np.array(float): An array of shape (2,), where the first 
        element is the probability of detection at A or C,
        and the second element is the probability of detection at D.
    """
    # Qubit 0 represents the photon's path: |0> for path 0, |1> for path 1.
    # Qubit 1 is an auxiliary qubit used as a flag for detector A's outcome.
    # Initially, both qubits are in |0>.

    # 1. Apply the first Beam Splitter (BS1) to the photon path (Qubit 0).
    # The photon starts in path |0>.
    qml.QubitUnitary(beam_splitter(r), wires=0)
    # State after BS1: (r|0> + sqrt(1-r^2)|1>)|0> (where Qubit 1 is still |0>)

    # 2. Model Detector A's effect and path selection.
    # If the photon is in path |0> (Qubit 0 is |0>), detector A detects it.
    # We use a CNOT gate with Qubit 0 as control and Qubit 1 as target.
    # If Qubit 0 is |0>, Qubit 1 remains |0> (signifying A detected).
    # If Qubit 0 is |1>, Qubit 1 flips to |1> (signifying A did NOT detect, and photon continues).
    qml.CNOT(wires=[0, 1])
    # State after CNOT: r|0>|0> + sqrt(1-r^2)|1>|1>
    # Interpretation:
    #   - |00> component corresponds to A detecting (photon in path 0, flag Q1 is 0).
    #   - |11> component corresponds to A NOT detecting (photon in path 1, flag Q1 is 1).

    # 3. Apply the second Beam Splitter (BS2) conditionally.
    # BS2 is applied ONLY if A did NOT detect (i.e., Qubit 1 is |1>).
    # We use qml.measure and qml.cond for this mid-circuit measurement and conditional operation.
    m_A_flag = qml.measure(1) # Symbolically measure Qubit 1

    # Define the operations for the true and false branches
    def apply_bs2_if_A_not_detected():
        # This function will be called if m_A_flag == 1
        qml.QubitUnitary(beam_splitter(r), wires=0)
    
    def do_nothing_if_A_detected():
        # This function will be called if m_A_flag == 0
        pass # No operation needed

    # Apply the conditional operation using the functional form of qml.cond
    # The result of qml.cond is an operation that needs to be "executed" by calling it
    qml.cond(m_A_flag == 1, apply_bs2_if_A_not_detected, do_nothing_if_A_detected)()
    
    # After the conditional operation, the full state is considered for probabilities:
    # P(|00>): Probability of A detecting
    # P(|01>): Probability of C detecting (given A did not)
    # P(|10>): Should be 0
    # P(|11>): Probability of D detecting (given A did not)

    # Return the probabilities of all basis states for the two qubits.
    return qml.probs(wires=[0, 1])


def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    
    # Get the probabilities of the final 2-qubit states from the QNode.
    # probs[0] = P(|00>) = Probability that A detects (photon in path 0, A detected)
    # probs[1] = P(|01>) = Probability that C detects (photon in path 0 after BS2, A did not detect)
    # probs[2] = P(|10>) = Should be ~0
    # probs[3] = P(|11>) = Probability that D detects (photon in path 1 after BS2, A did not detect)
    probs = mz_interferometer(ins)
    
    # Calculate the required probabilities:
    # p_AC: Probability that either A or C detect a photon. These are mutually exclusive.
    p_A = probs[0]
    p_C = probs[1]
    p_AC_final = p_A + p_C

    # p_D: Probability that D detects a photon.
    p_D_final = probs[3]
    
    # Convert the pennylane.numpy.tensor objects to standard Python floats
    # using .item() before JSON serialization.
    return json.dumps([p_AC_final.item(), p_D_final.item()])


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, atol=1e-6), "Not the correct probabilities"


test_cases = [
    ('0.5', '[0.8125, 0.1875]'),
    ('0.577350269', '[0.777778, 0.222222]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")



Running test case 0 with input '0.5'...
Correct!
Running test case 1 with input '0.577350269'...
Correct!
